
# ILLM Lab 7 — HMM Noun Phrase Chunking: Forward Algorithm & Viterbi Decoding

Trains a Hidden Markov Model to tag each word of a sentence as `B-NP`
(beginning of a noun phrase), `I-NP` (inside a noun phrase), or `O` (outside
any noun phrase) — the classic BIO chunking scheme, on CoNLL-2000-style data.
Both core algorithms are implemented **from scratch**:

1. **Forward algorithm** — the probability of each test sentence under the
   trained HMM.
2. **Viterbi decoding** — the single most likely tag sequence for each test
   sentence.

Evaluated with [`seqeval`](https://pypi.org/project/seqeval/), which scores
whole **chunks**, not individual tokens — getting 2 out of 3 words of a noun
phrase right still counts as getting that noun phrase *wrong*.

---

## 1. Deep analysis of the assignment

| # | Requirement | What it means concretely |
|---|---|---|
| 1 | Train/test data of (word, BIO-tag) pairs | Standard CoNLL chunking format: one `word<TAB>tag` per line, blank line between sentences. The 3 possible tags are the HMM's **hidden states**; the words are the **observations**. |
| 2 | Add-K smoothing for zero counts | Applied to *all three* probability tables the HMM needs — transition, emission, and initial-state — not just one of them (see Section 4). |
| 3 | Forward algorithm → $P(\text{sentence})$ | Sum, over every possible tag sequence, of $P(\text{tags}, \text{words})$ — computed by dynamic programming rather than actually enumerating every sequence, which is exponential in sentence length. |
| 4 | Viterbi decoding → predicted tags | The single **best** tag sequence, by the same style of dynamic programming but with $\max$ in place of $\text{sum}$, plus backpointers to recover the actual path. |
| 5 | Evaluate with `seqeval` | Chunk-level precision/recall/F1 — a predicted chunk only counts as correct if its **entire span and its `B`/`I` boundaries** match the gold chunk exactly, which token-level accuracy can't tell you. |

**Design decisions (stated explicitly, since the assignment leaves these open):**

- **Everything computed in log-space.** Raw probabilities are products of many
  numbers less than 1, and underflow to exactly `0.0` in floating point for
  even moderately long sentences — a real, not hypothetical, failure mode.
  Every formula below is written in plain probability notation (for
  readability), but implemented as sums of logs with the log-sum-exp trick
  wherever probabilities would otherwise need to be added.
- **Add-$K$ applies uniformly to transition, emission, *and* initial-state
  probabilities** — not just emissions. A test sentence starting with a word
  whose gold-standard first-tag was never seen as a *sentence-initial* tag in
  training, or containing a transition pattern never observed, would
  otherwise still break the model exactly the same way an unseen word would.
- **Out-of-vocabulary words are handled directly by the Add-$K$ formula**,
  not a separate `<UNK>` token: a word with zero training count for every tag
  still gets a valid, small, nonzero emission probability
  ($k / (\text{count}(tag) + kV)$ for every tag) automatically, with no
  special-casing needed.
- **$K$ is tuned**, not fixed to an arbitrary value: a held-out slice of the
  training sentences is scored with the forward algorithm itself (Section 6)
  under several candidate $K$ values, and the one maximizing held-out
  log-likelihood is kept — directly reusing Task 1's own algorithm to make
  the smoothing choice, rather than picking a number out of thin air.
- **The exact `train_NP.txt` / `test_NP.txt` files aren't available in this
  conversation**, so Section 2 is a loader that looks for them in the working
  directory and, if they're missing, falls back to a small built-in example
  built from the exact excerpt given in the assignment — enough to make
  every cell in this notebook runnable and verifiably correct without your
  real files.

## 2. Plan for this notebook

```
Load train_NP.txt / test_NP.txt (CoNLL format: word<TAB>tag per line)
        ▼
Count transitions, emissions, and sentence-initial tags on TRAIN only
        ▼
Add-K smoothed probability functions (Section 4) + a toy-corpus check:
   both algorithms are verified against BRUTE-FORCE enumeration of every
   possible tag sequence on a tiny example, before trusting them at scale
        ▼
Tune K on held-out training sentences, maximizing forward-algorithm
log-likelihood
        ▼
Task 1: forward algorithm -- P(sentence) for every test sentence
        ▼
Task 2: Viterbi decoding -- predicted tag sequence for every test sentence
        ▼
Evaluate with seqeval (chunk-level P/R/F1) + token-level accuracy for contrast
```



## Section 1 — Environment Setup

> **A real, reproducible install issue, not hypothetical:** plain
> `pip install seqeval` can fail with a `setuptools_scm` /
> `ModuleNotFoundError: No module named 'vcs_versioning'` build error on some
> environments (a packaging-toolchain incompatibility, unrelated to seqeval's
> own code). Installing `vcs_versioning` first fixes it. Without this fix,
> `pip` may silently fall back to a **very old** `seqeval` release (0.0.10,
> from ~2018) that depends on `Keras` and has a different API — worth
> knowing about if `pip install seqeval` alone ever "succeeds" but pulls in
> Keras as a dependency, since that's a sign it grabbed the wrong version.


In [ ]:

# !pip install --quiet vcs_versioning "seqeval>=1.2"


In [2]:

import os
import re
import math
import random
from collections import Counter

from seqeval.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

print("Environment ready.")


Environment ready.



## Section 2 — Load the Training and Test Data

**Point `TRAIN_PATH` / `TEST_PATH` at your real files** — CoNLL format,
`word<TAB>tag` per line, blank line between sentences. If they aren't found,
a small built-in example (built from the exact excerpt in the assignment,
extended slightly so it's actually usable) is used instead, purely so every
cell below is runnable and demonstrably correct without your files.


In [ ]:

TRAIN_PATH = "/Users/akshdharodiya/Desktop/SEM_7_LAB_PRACTICALS/LLM_LAB/LLM_LAB_7_10_09_26/Data/test_NP.txtk"
TEST_PATH = "/Users/akshdharodiya/Desktop/SEM_7_LAB_PRACTICALS/LLM_LAB/LLM_LAB_7_10_09_26/Data/test_NP.txt"

def parse_conll(path):
    """word<TAB>tag (or any whitespace) per line; blank line = sentence boundary."""
    sentences = []
    current = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.strip() == "":
                if current:
                    sentences.append(current)
                    current = []
                continue
            parts = line.split()
            if len(parts) < 2:
                continue
            word, tag = parts[0], parts[-1]
            current.append((word, tag))
    if current:
        sentences.append(current)
    return sentences

def build_demo_data():
    """Built from the assignment's own excerpt, extended with a few more
    sentences in the same style so training has enough signal to be
    meaningful. NOT a substitute for the real CoNLL NP-chunking files."""
    demo_train_text = """Rockwell\tB-NP
International\tI-NP
Corp.\tI-NP
's\tB-NP
Tulsa\tI-NP
unit\tI-NP
said\tO
it\tB-NP
signed\tO
a\tB-NP
tentative\tI-NP
agreement\tI-NP
with\tO
Boeing\tB-NP
Co.\tI-NP

The\tB-NP
company\tI-NP
declined\tO
to\tO
comment\tO
on\tO
the\tB-NP
matter\tI-NP

A\tB-NP
spokesman\tI-NP
for\tO
the\tB-NP
firm\tI-NP
said\tO
sales\tB-NP
rose\tO
sharply\tO
last\tB-NP
quarter\tI-NP

Investors\tB-NP
remain\tO
cautious\tO
about\tO
the\tB-NP
economic\tI-NP
outlook\tI-NP

The\tB-NP
pound\tI-NP
fell\tO
against\tO
the\tB-NP
dollar\tI-NP
yesterday\tO
"""
    demo_test_text = """Confidence\tB-NP
in\tO
the\tB-NP
pound\tI-NP
is\tO
widely\tO
expected\tO
to\tO
take\tO
another\tB-NP
sharp\tI-NP
dive\tI-NP

The\tB-NP
Tulsa\tI-NP
unit\tI-NP
declined\tO
to\tO
comment\tO

A\tB-NP
tentative\tI-NP
deal\tI-NP
was\tO
signed\tO
last\tB-NP
quarter\tI-NP
"""
    def parse_text(text):
        sentences, current = [], []
        for line in text.split("\n"):
            if line.strip() == "":
                if current:
                    sentences.append(current)
                    current = []
                continue
            parts = line.split("\t")
            if len(parts) == 2:
                current.append((parts[0], parts[1]))
        if current:
            sentences.append(current)
        return sentences
    return parse_text(demo_train_text), parse_text(demo_test_text)

if os.path.exists(TRAIN_PATH) and os.path.exists(TEST_PATH):
    train_sentences = parse_conll(TRAIN_PATH)
    test_sentences = parse_conll(TEST_PATH)
    print(f"Loaded real data from {TRAIN_PATH} / {TEST_PATH}.")
else:
    print("=" * 78)
    print("WARNING: train_NP.txt / test_NP.txt not found -- using the small built-in")
    print("DEMO dataset (from the assignment's own excerpt) instead. Save your real")
    print("files under those two names in this notebook's working directory, then")
    print("re-run this cell.")
    print("=" * 78)
    train_sentences, test_sentences = build_demo_data()

TAGS = sorted({tag for sent in train_sentences for _, tag in sent})
print(f"\nTags found: {TAGS}")
print(f"Train sentences: {len(train_sentences):,}   Test sentences: {len(test_sentences):,}")
print(f"\nExample training sentence:")
print(train_sentences[0])


Loaded real data from /Users/akshdharodiya/Desktop/SEM_7_LAB_PRACTICALS/LLM_LAB/LLM_LAB_7_10_09_26/Data/test_NP.txt / /Users/akshdharodiya/Desktop/SEM_7_LAB_PRACTICALS/LLM_LAB/LLM_LAB_7_10_09_26/Data/test_NP.txt.

Tags found: ['B-NP', 'I-NP', 'O']
Train sentences: 77   Test sentences: 77

Example training sentence:
[('Confidence', 'B-NP'), ('in', 'O'), ('the', 'B-NP'), ('pound', 'I-NP'), ('is', 'O'), ('widely', 'O'), ('expected', 'O'), ('to', 'O'), ('take', 'O'), ('another', 'B-NP'), ('sharp', 'I-NP'), ('dive', 'I-NP'), ('if', 'O'), ('trade', 'B-NP'), ('figures', 'I-NP'), ('for', 'O'), ('September', 'B-NP'), (',', 'O'), ('due', 'O'), ('for', 'O'), ('release', 'B-NP'), ('tomorrow', 'B-NP'), (',', 'O'), ('fail', 'O'), ('to', 'O'), ('show', 'O'), ('a', 'B-NP'), ('substantial', 'I-NP'), ('improvement', 'I-NP'), ('from', 'O'), ('July', 'B-NP'), ('and', 'I-NP'), ('August', 'I-NP'), ("'s", 'B-NP'), ('near-record', 'I-NP'), ('deficits', 'I-NP'), ('.', 'O')]



## Section 3 — Estimate HMM Parameters, With Add-$K$ Smoothing

An HMM for tagging needs three probability tables, all estimated by counting
on the training data:

**Initial-state probability** — how often each tag starts a sentence:
$$P(\text{tag}_1 = s) = \frac{\text{count}(s \text{ as first tag}) + K}{N_{\text{sentences}} + K \cdot |\text{TAGS}|}$$

**Transition probability** — how often tag $s$ follows tag $s'$:
$$P(\text{tag}_i = s \mid \text{tag}_{i-1} = s') = \frac{C(s', s) + K}{C(s') + K \cdot |\text{TAGS}|}$$

**Emission probability** — how often word $w$ is seen tagged as $s$:
$$P(w_i = w \mid \text{tag}_i = s) = \frac{C(s, w) + K}{C(s) + K \cdot V}$$

where $V$ is the training vocabulary size. **A word never seen in training at
all** (for any tag) still gets a valid probability from this same formula —
$C(s, w) = 0$ for every $s$, so it just becomes $K / (C(s) + KV)$ — no special
`<UNK>` handling is needed.


In [4]:

def fit_hmm(train_sents, tags, k):
    trans_counts = Counter()
    tag_counts = Counter()
    emit_counts = Counter()
    init_counts = Counter()
    vocab = set()

    for sent in train_sents:
        for i, (word, tag) in enumerate(sent):
            vocab.add(word)
            tag_counts[tag] += 1
            emit_counts[(tag, word)] += 1
            if i == 0:
                init_counts[tag] += 1
            else:
                prev_tag = sent[i - 1][1]
                trans_counts[(prev_tag, tag)] += 1

    V = len(vocab)
    n_tags = len(tags)
    n_sents = len(train_sents)

    def p_init(tag):
        return (init_counts[tag] + k) / (n_sents + k * n_tags)

    def p_trans(prev_tag, tag):
        return (trans_counts[(prev_tag, tag)] + k) / (tag_counts[prev_tag] + k * n_tags)

    def p_emit(tag, word):
        return (emit_counts[(tag, word)] + k) / (tag_counts[tag] + k * V)

    return {"p_init": p_init, "p_trans": p_trans, "p_emit": p_emit, "vocab": vocab, "V": V}



## Section 4 — Log-Space Forward Algorithm & Viterbi

$$\alpha_1(s) = P(\text{tag}_1{=}s)\,P(w_1\mid s) \qquad
\alpha_t(s) = \Big[\sum_{s'} \alpha_{t-1}(s')\,P(s\mid s')\Big] P(w_t\mid s) \qquad
P(\text{sentence}) = \sum_{s} \alpha_T(s)$$

$$\delta_1(s) = P(\text{tag}_1{=}s)\,P(w_1\mid s) \qquad
\delta_t(s) = \Big[\max_{s'} \delta_{t-1}(s')\,P(s\mid s')\Big] P(w_t\mid s)$$

Both are implemented with a `log_sum_exp` helper for the forward algorithm's
$\sum$ step (summing probabilities in log-space isn't just "take the log of
the sum" — it needs this specific trick to stay numerically stable), and with
a `max` (not `log_sum_exp`) for Viterbi's step, plus a backpointer at every
position to reconstruct the actual best path afterward.


In [5]:

def log_sum_exp(values):
    m = max(values)
    if m == float("-inf"):
        return float("-inf")
    return m + math.log(sum(math.exp(v - m) for v in values))


def forward_logprob(words, tags, hmm):
    p_init, p_trans, p_emit = hmm["p_init"], hmm["p_trans"], hmm["p_emit"]
    alpha = {s: math.log(p_init(s)) + math.log(p_emit(s, words[0])) for s in tags}
    for w in words[1:]:
        new_alpha = {}
        for s in tags:
            terms = [alpha[sp] + math.log(p_trans(sp, s)) for sp in tags]
            new_alpha[s] = log_sum_exp(terms) + math.log(p_emit(s, w))
        alpha = new_alpha
    return log_sum_exp(list(alpha.values()))


def viterbi_decode(words, tags, hmm):
    p_init, p_trans, p_emit = hmm["p_init"], hmm["p_trans"], hmm["p_emit"]
    # delta[s] = (best log-prob ending in state s so far, best path ending in s so far)
    delta = {s: (math.log(p_init(s)) + math.log(p_emit(s, words[0])), [s]) for s in tags}
    for w in words[1:]:
        new_delta = {}
        for s in tags:
            best_prev = max(tags, key=lambda sp: delta[sp][0] + math.log(p_trans(sp, s)))
            score = delta[best_prev][0] + math.log(p_trans(best_prev, s)) + math.log(p_emit(s, w))
            new_delta[s] = (score, delta[best_prev][1] + [s])
        delta = new_delta
    best_final_tag = max(tags, key=lambda s: delta[s][0])
    best_logprob, best_path = delta[best_final_tag]
    return best_logprob, best_path



## Section 5 — Toy-Corpus Validation: Both Algorithms Against Brute Force

The strongest possible correctness check for both algorithms: on a small
enough example, actually **enumerate every possible tag sequence**, compute
its true probability directly (no dynamic programming shortcuts), and confirm
the forward algorithm's total matches the brute-force sum exactly, and
Viterbi's chosen path and its probability match the brute-force maximum
exactly — including on a sentence with a word never seen in training.


In [6]:

from itertools import product

toy_train = [
    [("the", "B-NP"), ("cat", "I-NP"), ("sat", "O")],
    [("a", "B-NP"), ("dog", "I-NP"), ("ran", "O"), ("fast", "O")],
    [("the", "B-NP"), ("big", "I-NP"), ("dog", "I-NP"), ("barked", "O")],
]
toy_tags = ["O", "B-NP", "I-NP"]
toy_hmm = fit_hmm(toy_train, toy_tags, k=0.5)

def brute_force_logprob(words, tags, hmm):
    total = 0.0
    for path in product(tags, repeat=len(words)):
        p = hmm["p_init"](path[0]) * hmm["p_emit"](path[0], words[0])
        for i in range(1, len(words)):
            p *= hmm["p_trans"](path[i - 1], path[i]) * hmm["p_emit"](path[i], words[i])
        total += p
    return math.log(total)

def brute_force_viterbi(words, tags, hmm):
    best_p, best_path = -1.0, None
    for path in product(tags, repeat=len(words)):
        p = hmm["p_init"](path[0]) * hmm["p_emit"](path[0], words[0])
        for i in range(1, len(words)):
            p *= hmm["p_trans"](path[i - 1], path[i]) * hmm["p_emit"](path[i], words[i])
        if p > best_p:
            best_p, best_path = p, path
    return math.log(best_p), list(best_path)

toy_test_words = [
    ["the", "dog", "sat"],
    ["a", "big", "cat", "ran"],
    ["the", "unseen_word", "barked"],  # deliberately out-of-vocabulary
]

all_ok = True
for words in toy_test_words:
    fwd = forward_logprob(words, toy_tags, toy_hmm)
    bf_fwd = brute_force_logprob(words, toy_tags, toy_hmm)
    vit_score, vit_path = viterbi_decode(words, toy_tags, toy_hmm)
    bf_score, bf_path = brute_force_viterbi(words, toy_tags, toy_hmm)

    fwd_ok = abs(fwd - bf_fwd) < 1e-9
    vit_ok = abs(vit_score - bf_score) < 1e-9 and vit_path == bf_path
    sanity_ok = vit_score <= fwd + 1e-9  # a single path can never beat the sum over ALL paths
    all_ok &= fwd_ok and vit_ok and sanity_ok

    print(f"words={words}")
    print(f"  forward: dp={fwd:.6f} brute-force={bf_fwd:.6f} [{'OK' if fwd_ok else 'FAIL'}]")
    print(f"  viterbi: dp={vit_score:.6f} brute-force={bf_score:.6f} path_match={vit_path==bf_path} [{'OK' if vit_ok else 'FAIL'}]")
    print(f"  sanity (viterbi <= forward): {sanity_ok}")
    print()

print("ALL CHECKS PASSED" if all_ok else "SOME CHECKS FAILED -- stop and debug before scaling up")
assert all_ok


words=['the', 'dog', 'sat']
  forward: dp=-4.787619 brute-force=-4.787619 [OK]
  viterbi: dp=-5.011603 brute-force=-5.011603 path_match=True [OK]
  sanity (viterbi <= forward): True

words=['a', 'big', 'cat', 'ran']
  forward: dp=-8.387678 brute-force=-8.387678 [OK]
  viterbi: dp=-9.067138 brute-force=-9.067138 path_match=True [OK]
  sanity (viterbi <= forward): True

words=['the', 'unseen_word', 'barked']
  forward: dp=-6.276368 brute-force=-6.276368 [OK]
  viterbi: dp=-6.621041 brute-force=-6.621041 path_match=True [OK]
  sanity (viterbi <= forward): True

ALL CHECKS PASSED



## Section 6 — Tune $K$ on Held-Out Data

A slice of the *training* sentences (not test — test stays untouched until
Section 8) is held out, an HMM is fit on the rest for each candidate $K$, and
the forward algorithm itself (Section 4) scores the held-out sentences —
$K$ is chosen by whichever value gives the **highest held-out total
log-likelihood** (equivalently, lowest perplexity). This directly reuses
Task 1's own algorithm to make the smoothing decision, rather than picking
$K$ arbitrarily.


In [7]:

_shuffled = train_sentences[:]
random.Random(RANDOM_SEED).shuffle(_shuffled)
_split = max(1, int(len(_shuffled) * 0.85))
_tune_train, _tune_heldout = _shuffled[:_split], _shuffled[_split:]

K_CANDIDATES = [0.0001, 0.001, 0.01, 0.05, 0.1, 0.2, 0.5, 1.0]

k_results = []
for k in K_CANDIDATES:
    hmm_k = fit_hmm(_tune_train, TAGS, k)
    total_ll, total_tokens = 0.0, 0
    for sent in _tune_heldout:
        words = [w for w, _ in sent]
        total_ll += forward_logprob(words, TAGS, hmm_k)
        total_tokens += len(words)
    perplexity = math.exp(-total_ll / total_tokens) if total_tokens else float("inf")
    k_results.append((k, total_ll, perplexity))
    print(f"k={k:<8} held-out total log-likelihood={total_ll:10.3f}  perplexity={perplexity:.3f}")

BEST_K = max(k_results, key=lambda r: r[1])[0]
print(f"\nChosen K = {BEST_K}")


k=0.0001   held-out total log-likelihood= -2861.599  perplexity=3393.251
k=0.001    held-out total log-likelihood= -2626.940  perplexity=1742.189
k=0.01     held-out total log-likelihood= -2394.149  perplexity=899.249
k=0.05     held-out total log-likelihood= -2239.470  perplexity=579.480
k=0.1      held-out total log-likelihood= -2180.572  perplexity=490.196
k=0.2      held-out total log-likelihood= -2132.277  perplexity=427.351
k=0.5      held-out total log-likelihood= -2095.571  perplexity=385.032
k=1.0      held-out total log-likelihood= -2094.793  perplexity=384.182

Chosen K = 1.0


In [8]:

# Final HMM: refit on ALL training data (not just the tuning subset) with the chosen K
hmm = fit_hmm(train_sentences, TAGS, BEST_K)
print(f"Final HMM fit on all {len(train_sentences):,} training sentences with K={BEST_K}.")
print(f"Vocabulary size: {hmm['V']:,}")


Final HMM fit on all 77 training sentences with K=1.0.
Vocabulary size: 744



## Section 7 — Task 1: Forward Algorithm on the Test Set

$P(\text{sentence})$ for every test sentence. Raw probabilities underflow to
`0.0` almost immediately for real sentences, so both the log-probability and
**perplexity per word** (a length-normalized, more directly comparable
number: $\exp(-\text{logprob}/\text{length})$) are reported.


In [9]:

forward_results = []
for sent in test_sentences:
    words = [w for w, _ in sent]
    logprob = forward_logprob(words, TAGS, hmm)
    perplexity = math.exp(-logprob / len(words))
    forward_results.append({"sentence": " ".join(words), "length": len(words),
                             "log P(sentence)": logprob, "perplexity/word": perplexity})

import pandas as pd
forward_df = pd.DataFrame(forward_results)
pd.set_option("display.max_colwidth", 60)
forward_df


,sentence,length,log P(sentence),perplexity/word
0,Confidence in the pound is widely expected to take anoth...,37,-215.951563,342.588061
1,Chancellor of the Exchequer Nigel Lawson 's restated com...,27,-156.024137,323.329436
2,But analysts reckon underlying support for sterling has ...,29,-177.713670,458.544604
3,This has increased the risk of the government being forc...,36,-203.981229,288.918674
4,`` The risks for sterling of a bad trade figure are very...,31,-179.617693,328.362810
...,...,...,...,...
72,"Less-than-truckload companies , which carry the freight ...",26,-152.929494,358.491028
73,The carriers were competing fiercely for market share .,9,-53.891925,398.613261
74,Railroad-rate increases are likely to be restrained by w...,19,-113.904836,401.413229
75,"An official at Consolidated Freightways Inc. , a Menlo P...",29,-164.251663,288.256427



## Section 8 — Task 2: Viterbi Decoding on the Test Set

The most likely tag sequence for every test sentence, using the **gold
words only** — the true tags are never given to the decoder, only kept aside
for Section 9's evaluation.


In [10]:

predicted_tags = []
gold_tags = []
for sent in test_sentences:
    words = [w for w, _ in sent]
    true_tags = [t for _, t in sent]
    _, pred_path = viterbi_decode(words, TAGS, hmm)
    predicted_tags.append(pred_path)
    gold_tags.append(true_tags)

print("Example (first test sentence):")
words0 = [w for w, _ in test_sentences[0]]
print(f"{'Word':<15}{'Gold':<10}{'Predicted'}")
for w, g, p in zip(words0, gold_tags[0], predicted_tags[0]):
    flag = "" if g == p else "  <-- mismatch"
    print(f"{w:<15}{g:<10}{p}{flag}")


Example (first test sentence):
Word           Gold      Predicted
Confidence     B-NP      B-NP
in             O         O
the            B-NP      B-NP
pound          I-NP      I-NP
is             O         O
widely         O         O
expected       O         O
to             O         O
take           O         O
another        B-NP      B-NP
sharp          I-NP      I-NP
dive           I-NP      I-NP
if             O         O
trade          B-NP      B-NP
figures        I-NP      I-NP
for            O         O
September      B-NP      B-NP
,              O         O
due            O         O
for            O         O
release        B-NP      B-NP
tomorrow       B-NP      I-NP  <-- mismatch
,              O         O
fail           O         O
to             O         O
show           O         O
a              B-NP      B-NP
substantial    I-NP      I-NP
improvement    I-NP      I-NP
from           O         O
July           B-NP      B-NP
and            I-NP      I-NP
August  


## Section 9 — Evaluation with `seqeval`

`seqeval` scores whole **chunks** (a `B-NP` tag plus every `I-NP` tag that
follows it, until the next `O` or `B-NP`), not individual tokens — a
predicted chunk only counts as correct if its span and every tag in it match
the gold chunk exactly.


In [11]:

print(classification_report(gold_tags, predicted_tags))

chunk_f1 = f1_score(gold_tags, predicted_tags)
chunk_precision = precision_score(gold_tags, predicted_tags)
chunk_recall = recall_score(gold_tags, predicted_tags)
print(f"Chunk-level F1:        {chunk_f1:.4f}")
print(f"Chunk-level Precision: {chunk_precision:.4f}")
print(f"Chunk-level Recall:    {chunk_recall:.4f}")


              precision    recall  f1-score   support

          NP       0.80      0.79      0.80       475

   micro avg       0.80      0.79      0.80       475
   macro avg       0.80      0.79      0.80       475
weighted avg       0.80      0.79      0.80       475

Chunk-level F1:        0.7958
Chunk-level Precision: 0.8000
Chunk-level Recall:    0.7916


In [12]:

# Token-level accuracy, for contrast with the chunk-level numbers above --
# NOT a substitute for the seqeval scores, just useful context for them.
total_tokens = sum(len(g) for g in gold_tags)
correct_tokens = sum(1 for g_seq, p_seq in zip(gold_tags, predicted_tags)
                      for g, p in zip(g_seq, p_seq) if g == p)
token_accuracy = correct_tokens / total_tokens
print(f"Token-level accuracy: {token_accuracy:.4f}  ({correct_tokens}/{total_tokens} tokens)")
print(f"Chunk-level F1:       {chunk_f1:.4f}")
print("\nA gap between these two numbers is expected and informative -- see the")
print("discussion below.")


Token-level accuracy: 0.9293  (1762/1896 tokens)
Chunk-level F1:       0.7958

A gap between these two numbers is expected and informative -- see the
discussion below.



## Section 10 — Qualitative Examples

A handful of test sentences with gold vs. predicted tags side by side —
useful for spotting *what kind* of mistakes the model makes (off-by-one
boundaries? whole noun phrases missed entirely? spurious short chunks?),
which the aggregate numbers above can't show on their own.


In [13]:

N_EXAMPLES = min(5, len(test_sentences))
for i in range(N_EXAMPLES):
    words = [w for w, _ in test_sentences[i]]
    print(f"Sentence {i}: " + " ".join(words))
    print(f"  {'Word':<15}{'Gold':<10}{'Predicted'}")
    any_mismatch = False
    for w, g, p in zip(words, gold_tags[i], predicted_tags[i]):
        flag = "" if g == p else "  <-- mismatch"
        if flag:
            any_mismatch = True
        print(f"  {w:<15}{g:<10}{p}{flag}")
    if not any_mismatch:
        print("  (perfect match)")
    print()


Sentence 0: Confidence in the pound is widely expected to take another sharp dive if trade figures for September , due for release tomorrow , fail to show a substantial improvement from July and August 's near-record deficits .
  Word           Gold      Predicted
  Confidence     B-NP      B-NP
  in             O         O
  the            B-NP      B-NP
  pound          I-NP      I-NP
  is             O         O
  widely         O         O
  expected       O         O
  to             O         O
  take           O         O
  another        B-NP      B-NP
  sharp          I-NP      I-NP
  dive           I-NP      I-NP
  if             O         O
  trade          B-NP      B-NP
  figures        I-NP      I-NP
  for            O         O
  September      B-NP      B-NP
  ,              O         O
  due            O         O
  for            O         O
  release        B-NP      B-NP
  tomorrow       B-NP      I-NP  <-- mismatch
  ,              O         O
  fail           O   


## Section 11 — Discussion

Guidance for what to look for and write up, grounded in what each piece of
this pipeline is built to do — fill in with the *actual* numbers from your
run:

- **Chunk-level F1 (seqeval) is usually noticeably lower than token-level
  accuracy**, and that gap is not a bug — it's the entire reason `seqeval`
  exists instead of just measuring per-token correctness. A single wrong tag
  at a noun-phrase boundary (predicting `O` where the gold says `I-NP`, or
  splitting one long chunk into two shorter ones) costs an *entire chunk* of
  credit under chunk-level scoring, while only costing one token's worth
  under token accuracy. Report both, and note the gap explicitly rather than
  only citing whichever number looks better.
- **The chosen $K$** (Section 6) is worth reporting alongside *why* the held-out
  log-likelihood favored it — very small $K$ tends to win when training data
  is large/repetitive relative to the tag set (little true sparsity to guard
  against), while a larger $K$ typically wins on smaller or more diverse
  training sets, where genuine held-out novelty is common enough that
  aggressive smoothing pays off. Compare BEST_K against how big your
  training set actually is.
- **The Section 5 sanity check (Viterbi log-prob ≤ forward log-prob) is a
  built-in invariant of any correct HMM implementation** — the forward
  algorithm sums over *every* tag path, so no single path (including
  Viterbi's best one) can ever have higher probability than that sum. Worth
  re-running that exact check against a small slice of your *real* data
  (not just the toy corpus) as an extra confidence check before trusting the
  test-set numbers.
- **Where do the errors cluster?** Look at Section 10's mismatches: are they
  concentrated at chunk *boundaries* (predicting `B-NP` one word early/late,
  or confusing `B-NP`/`I-NP` right after a genuine chunk start), or does the
  model sometimes miss entire noun phrases (predicting `O` throughout)? The
  former suggests the transition probabilities need more training data or a
  different $K$; the latter more often points to emission-side sparsity —
  content words the model essentially never saw during training.
